In [3]:
from google.colab import drive
import numpy as np


In [4]:
drive.mount("/content/drive", force_remount = True)

OUTPUT_DIR = "/content/drive/MyDrive/neurosynth/data/processed"

X = np.load(f"{OUTPUT_DIR}/X.npy")
y = np.load(f"{OUTPUT_DIR}/y.npy")


print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Left fist: {(y==0).sum()}, Right fist: {(y ==1).sum()}")

Mounted at /content/drive
X shape: (225, 64, 641)
y shape: (225,)
Left fist: 113, Right fist: 112


In [5]:
!pip install torch scikit-learn -q

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.11.0+cpu


In [7]:
# Split into train/test sets

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split (
    X, y,
    test_size = 0.2,
    stratify = y,
    random_state = 42
)


# stratify = y
"""
 our classes are nearly 50/50 (113 left, 112 right) without
stratify, the random split could accidentally put, 35 left first
epochs in training but only 10 in testing
an uneven split



but the stratify = y tells,  keep the same left/right ratio
in BOTH the train set and test.
so it overall it's 50/50 train will be, 50/50 and TEST will be 50/50 t00
"""

"\n our classes are nearly 50/50 (113 left, 112 right) without\nstratify, the random split could accidentally put, 35 left first\nepochs in training but only 10 in testing\nan uneven split\n\n\n\nbut the stratify = y tells,  keep the same left/right ratio\nin BOTH the train set and test.\nso it overall it's 50/50 train will be, 50/50 and TEST will be 50/50 t00\n"

In [8]:
print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

print(f"\nTrain - Left: {(y_train ==0).sum()},   Right: {(y_train==1).sum()}")
print(f"Test - Left:{(y_test ==0).sum()},     Right: {(y_test == 1).sum()}")

Train set: 180 samples
Test set: 45 samples

Train - Left: 90,   Right: 90
Test - Left:23,     Right: 22


In [9]:
# Wrap data in PyTorch dataset
class EEGDataset(Dataset):
  """
  every PyTorch Dataset needs exactly 3 methods.

  __init__ --> setup, store the data
  __len__  --> tell PyTorch how many samples exist
  __getitem__ --> return one sample given an index


  """
  def __init__(self, X, y):
    # converting numpy array into a PyTorch tensor
    # floatTensor because EEG values are decimal
    self.X = torch.FloatTensor(X)


    # LongTensor because labels are class indices( 0 or 1)
    # PyTorch's loss functions require this type
    # converting the numpy array(labels here) to whole numbers, not decimals (Pytorch tensor)
    self.y = torch.LongTensor(y)

  def __len__(self):
    # total number of samples - used internally by DataLoader
    return len(self.X)

  def __getitem__(self, idx):
    # return one sample (signal + label) at position idx
    return self.X[idx], self.y[idx]


In [10]:
train_dataset = EEGDataset(X_train, y_train)
test_dataset = EEGDataset(X_test, y_test)



print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size:  {len(test_dataset)}")

Train dataset size: 180
Test dataset size:  45


In [11]:
# Wrap dataset in a DataLoader
"""
the Dataset can hand us ONE sample. if we ask for it directly. But training a model on
one sample at a time is slow and noisy. We want to deed the model small groupd os
samples together - called a batch. so it can learn more stably and efficiently.

without DataLoader: model sees 1 sample -> updates -> 1 sample -> updates

"""


# how many samples the model sees at once before updating
BATCH_SIZE = 16


train_loader = DataLoader (
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True  # shuffle order every epoch - prevents
                    # the model from memorizing data order
)

"""
shuffle = True ; mixes up the order every single epoch (every
full pass through the data), so the model sees a random mix of
left/right in every batch - preventing it from learning the
order of the data instead of actual patterns

"""

test_loader = DataLoader (
    test_dataset,
    batch_size = BATCH_SIZE,
    shuffle = False

)

In [12]:
train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle =True)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False)



print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 12
Test batches:  3


In [13]:
sample_X, sample_y = next(iter(train_loader))
print(f"\nBatch X shape: {sample_X.shape}")
print(f"                          (batch_size, channels, timepoints)")
print(f"Batch y shape:   {sample_y.shape}")
print(f"Batch y values:   {sample_y}")


Batch X shape: torch.Size([16, 64, 641])
                          (batch_size, channels, timepoints)
Batch y shape:   torch.Size([16])
Batch y values:   tensor([0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0])


In [14]:
# Building the model Architecture
# 1. Reshape data so time = squence dimensions

# A currently has shape (n_samples, 64, 641)
# we need shape(n_samples, 641, 64)
# so we swap the LAST two dimensions

X_train_t = X_train.transpose(0, 2, 1)
X_test_t = X_test.transpose(0, 2, 1)

print(f"Before: {X_train.shape}")
print(f"After: {X_train_t.shape}")

Before: (180, 64, 641)
After: (180, 641, 64)


In [15]:
# Update the Dataset to use transposed Data

train_dataset = EEGDataset(X_train_t, y_train)
test_dataset = EEGDataset(X_test_t, y_test)

In [16]:
# Update the DataLoader to use transposed Data
train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False)

In [17]:
# verifying the changes (in transposed data)
sample_X, sample_y = next(iter(train_loader))
print(f"Batch X Shape: {sample_X.shape}")
print(f"Batch y Shape: {sample_y.shape}")

Batch X Shape: torch.Size([16, 641, 64])
Batch y Shape: torch.Size([16])


In [18]:
import torch.nn as nn

conv_layer = nn.Conv1d(
    in_channels = 64, # how many numbers describe each position in my input sequence
    out_channels = 32, # how many different pattern detectors (filters)
    kernel_size = 5,
    padding= 2
)

In [19]:
# Testing the conv layer
# pull one batch from our train_loader

sample_X, sample_y = next(iter(train_loader))
print(f"Original shape: {sample_X.shape}")


# Swap dimensions just for the conv layer
# Conv1d wants (batch, channels, time) --> so we swap dims 1 and 2
sample_X_for_conv = sample_X.transpose(1,2)
print(f"Shape for conv: {sample_X_for_conv.shape}")

Original shape: torch.Size([16, 641, 64])
Shape for conv: torch.Size([16, 64, 641])


In [20]:
# Actually run it through the conv layer
output = conv_layer(sample_X_for_conv)
print(f"Output shape: {output.shape}")

Output shape: torch.Size([16, 32, 641])


In [21]:
# Building 3 conv layers at different scales
# we already have one conv layer (window = 5)
# adding two more (medium + large windows)

# padding = (kernel_size - 1) /2

# small scale - catches fast, fine grained patterns
conv_small = nn.Conv1d(in_channels = 64, out_channels = 32, kernel_size = 5, padding = 2)

# medium scale - catches mid - range patterns
conv_medium = nn.Conv1d(in_channels = 64, out_channels = 32, kernel_size = 25, padding = 12)

# large scale - catches slow, broad trends
conv_large = nn.Conv1d(in_channels = 64, out_channels = 32, kernel_size = 75, padding = 37)





output_small = conv_small(sample_X_for_conv)
output_medium = conv_medium(sample_X_for_conv)
output_large = conv_large(sample_X_for_conv)


print(f"Small scale output: {output_small.shape}")
print(f"Medium scale output: {output_medium.shape}")
print(f"Large scale output:  {output_large.shape}")

Small scale output: torch.Size([16, 32, 641])
Medium scale output: torch.Size([16, 32, 641])
Large scale output:  torch.Size([16, 32, 641])


In [22]:
# combine all three scales together

combined = torch.cat([output_small, output_medium, output_large], dim =1)
print(f"Combined shape: {combined.shape}")

Combined shape: torch.Size([16, 96, 641])


In [23]:
class MultiScaleConv(nn.Module):
  """
  Extract local temporal patterns at 3 different time scales,
  then combines them into one richer feature representation
  """

  def __init__(self, in_channels = 64, out_channels = 32):
    super().__init__()

    self.conv_small = nn.Conv1d(in_channels, out_channels, kernel_size = 5, padding = 2)
    self.conv_medium = nn.Conv1d(in_channels, out_channels, kernel_size = 25, padding = 12)
    self.conv_large = nn.Conv1d(in_channels, out_channels, kernel_size = 75, padding = 37)


  def forward(self, x):
    # x comes in as (batch, time, channels) - our natural shape
    # Conv1d needs (batch, channels, time) so transpose first

    x = x.transpose(1,2)

    out_small = self.conv_small(x)
    out_medium = self.conv_medium(x)
    out_large = self.conv_large(x)


    # combine all 3 scales along the feature dimenion
    combined = torch.cat([out_small, out_medium, out_large], dim = 1)


    # transpose back to (batch, time, features) for the Transformer
    combined = combined.transpose(1,2)

    return combined

In [24]:
multi_scale = MultiScaleConv(in_channels = 64, out_channels =32)

sample_X, sample_y = next(iter(train_loader))
output = multi_scale(sample_X)


print(f"Input shape: {sample_X.shape}")
print(f"Output shape: {output.shape}")

Input shape: torch.Size([16, 641, 64])
Output shape: torch.Size([16, 641, 96])


In [25]:
# Attention (Q, K, V)
# Transformer Encoder
encoder_layer = nn.TransformerEncoderLayer(
    d_model = 96,
    nhead = 4,
    dim_feedforward = 256,
    batch_first = True

)

In [26]:
# Stack multiple encoder layers
transformer_encoder = nn.TransformerEncoder(
    encoder_layer,
    num_layers = 2
)

In [27]:
encoded_output = transformer_encoder(output) # output from our MultiScaleConv test
print(f"Transformer output shape: {encoded_output.shape}")

Transformer output shape: torch.Size([16, 641, 96])


In [28]:
# Classification head

class ClassificationHead(nn.Module):
  """
  takes the transformer's output and produces
  a final left/right classifiation score.

  """

  def __init__(self, d_model = 96, num_classes = 2):
    super().__init__()

    # linear layer: maps 96 features -- 2 class scores
    # num_classes = 2 because we have left fitst and right fist
    self.linear = nn.Linear(d_model, num_classes)

  def forward(self, x):
     # x shape coming in: (batch, 641, 96)

     # step 1: average acrosss the time dimension (dim =1)
     # collapses 641 timepoints into one summary vector
     x = x.mean(dim = 1)
     # x shape now: (batch, 96)


     # step 2: linear layer produces 2 scores
     x = self.linear(x)
     # x shape now: (batch, 2)


     return x

In [29]:
head = ClassificationHead(d_model = 96, num_classes = 2)

# use the encoded_output from our Transformer test above
final_output = head(encoded_output)
print(f"Classification head output: {final_output.shape}")

Classification head output: torch.Size([16, 2])


In [30]:
# Assemble all 3 together
class EEGTransformer(nn.Module):
  """
  full model pipeline:
  EEG signal -> MultiScaleConv -> Transformer -> ClassificationHead --> Prediction

  """
  def __init__(
      self,
      in_channels = 64,     # number of EEG channels
      conv_out = 32,        # filters per conv scale
      d_model = 96,         # feature size (32 * 3 scales = 96)
      nhead = 4,            # attention heads
      num_layers = 2,       # transformer layers
      num_classes = 2       # left fist or right fist
  ):

      super().__init__()


      # Step 01: extract multi-scale local temporal patterns
      self.multi_scale_conv = MultiScaleConv(
          in_channels = in_channels,
          out_channels = conv_out
      )


      # Step 02: model long-range global relationships across time
      encoder_layer = nn.TransformerEncoderLayer(
          d_model = d_model,
          nhead = nhead,
          dim_feedforward = 256,
          batch_first = True,     # expects shape: (batch, seq, feature)
          dropout = 0.1           # drops 10% of activations to prevent overfitting
      )


      self.transformer = nn.TransformerEncoder(
          encoder_layer,
          num_layers = num_layers

      )


      #Step 03: pool features and map to final class scores
      self.head = ClassificationHead(
          d_model = d_model,
          num_classes = num_classes
      )



  def forward(self, x):
    # x shape incoming: (batch, 641, 64)

    # 1. Local feature extraction
    x = self.multi_scale_conv(x)
    # x shape now: (batch, 641, 96)


    # 2. Global sequence modeling
    x = self.transformer(x)
    # x shape now: (batch, 641, 96)


    # 3. Final classification decision
    x = self.head(x)
    # x shape now: (batch, 2)



    return x


In [31]:
model = EEGTransformer(
    in_channels = 64,
    conv_out = 32,
    d_model = 96,
    nhead = 4,
    num_layers = 2,
    num_classes = 2
)


predictions = model(sample_X)
print(f"Input shape: {sample_X.shape}")
print(f"Output shape: {predictions.shape}")

Input shape: torch.Size([16, 641, 64])
Output shape: torch.Size([16, 2])


In [33]:
# LOSS FUNCTION + OPTIMIZER

# 1.0 set up device
# check if google colab allocated a GPU to this notebook
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# 2.0 create model and move to target hardware
model = EEGTransformer().to(device)
# .to(device) recursively copies all weight tensors from system RAM into GPU VRAM

# 3.0 define the loss function
# criterion that evaluates logits directly vs target ground-truth indices (0 or 1)
criterion = nn.CrossEntropyLoss()

# 4.0 instantiate optimizer
# model parameters pass into Adam so it tracks and modifies gradients directly
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)


print("Model loaded onto hardware accelerator")
print("Loss function initialized: CrossEntropyLoss")
print("Optimizer initialized: Adam (lr = 0.001)")

Using device: cuda
Model loaded onto hardware accelerator
Loss function initialized: CrossEntropyLoss
Optimizer initialized: Adam (lr = 0.001)


In [1]:
# Training and Evaluation
def train_one_epoch(model, loader, criterion, optimizer, device):
  """
  runs ONE full pass through the training data.
  updates weights and returns average loss and accuracy
  for this epoch

  """

  # 1. set model to training mode (enables dropout)
  model.train()

  total_loss = 0.0
  correct = 0
  total = 0


  # Iterate over mini-batches from the DataLoader
  for batch_X, batch_y in loader:
    # move data tensors to the taregt compute hardware
    batch_X = batch_X.to(device) # shape: (batch_size, 641, 64)
    batch_y = batch_y.to(device) # shape. (batch_size, )


    # Step A: forward pass
    predictions = model(batch_X)  # shape: (batch_size, 2)

    # Step B: Calculate  loss
    loss = criterion(predictions, batch_y)

    # Step C: backward pass
    optimizer.zero_grad()   # reset gradients from previous batch
    loss.backward()         # backpropagate errors to calculate new gradients
    optimizer.step()        # update weights based on gradients


    # Step D: Track metrics
    total_loss += loss.items()


    # extract  index of higher class score: dim = 1 chooses max across class dimention
    predicted_classes = predictions.argmax(dim = 1) # shape: (batch_size, )
    correct += (predicted_classes == batch_y).sum().item()
    total += batch_y.size(0)

  avg_loss = total_loss/ len(loader)
  accuracy = (correct/ total) * 100.0

  return avg_loss, accuracy


def evaluate(model, loader, criterion, device):
  """
  evaluates the model on validation/test data
  no weight updates occur - purely measures model generalization

  """


  # 1. set model to evaluation mode (disables dropout)
  model.eval()


  total_loss = 0.0
  correct = 0
  total = 0

  # Disable gradient computation graph to save memory and compute
  # torch.no_grad() tells PyTorch not to track gradients
  # we don't need them during evaluation — saves memory and is faster

  with torch.no_grad():
    for batch_X, batch_y in loader:
      batch_X = batch_X.to(device)
      batch_y = batch_y.to(device)



      # Forward pass
      predictions = model(batch_X)
      loss = criterion(predictions, batch_y)


      # Track metrics
      total_loss += loss.item()
      predicted_classes = predictions.argmax(dim = 1)
      correct += (predicted_classes == batch_y).sum().item()

      total += batch_y.size(0)



    # compute validation/test statistis
    avg_loss = total_loss /len(loader)
    accuracy = (correct / total) * 100.0


    return avg_loss, accuracy





In [2]:
import json
# shutil = copy files between folders
import shutil
import os
import subprocess

# datetime = get current date/time for commit messages
from datetime import datetime
from google.colab import userdata


GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
GITHUB_USERNAME = "the-liyanage"
REPO_NAME = "neurosynth"


# 1. set git identity
subprocess.run(["git", "config", "--global",
               "user.name", "the-liyanage"])

subprocess.run(["git", "config", "--global",
                "user.email", "hiruniliyanage4@gmail.com"])


# 2. Clone repo fresh (only if not already there)
if not os.path.exists(f"/content/{REPO_NAME}/.git"):
  subprocess.run(["rm", "-rf", "f/content/{REPO_NAME}"])
  os.chdir("/content")
  subprocess.run([
      "git", "clone",
      f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
      ])
  print("Repo clones fresh!")
else:
  print("Repo already exisit, skipping clone")




# 3. Copy notebook into repo
os.makedirs(f"/content/{REPO_NAME}/notebooks", exist_ok = True)
shutil.copy(
    "/content/drive/MyDrive/Colab Notebooks/03_model_training.ipynb",
    f"/content/neurosynth/notebooks/03_model_training.ipynb"
)
print(f"\n Notebook copied!")


# 4. Commit and push

# move into the repo folder
os.chdir(f"/content/{REPO_NAME}")

# stage all change
subprocess.run(["git", "pull"])
subprocess.run(["git", "add", "."])


commit_message = "training the model"

subprocess.run(["git", "commit", "-m", commit_message])


result = subprocess.run([
    "git", "push",
    f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git",
    "main"
], capture_output=True, text=True)


if result.returncode == 0:
  print("Pushed to the github", commit_message)
else:
  print("Push failed")
  print(result.stderr)

Repo clones fresh!


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/03_model_training.ipynb'